In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [9]:
import os

print("TRACING:", os.getenv("LANGSMITH_TRACING"))
print("PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("API KEY EXISTS:", bool(os.getenv("LANGSMITH_API_KEY")))

TRACING: true
PROJECT: lca-lc-langchain
API KEY EXISTS: True


In [10]:
from langsmith import traceable

@traceable
def test():
    return "hello"

test()

'hello'

In [11]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [12]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

groq_model = ChatGroq(model="llama-3.1-8b-instant") 
agent = create_agent(
    model=groq_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [14]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Here are some suggestions for recipes that can be made with leftover chicken and rice:

1. Chicken and Rice Casserole
2. Chicken Fried Rice
3. Chicken and Rice Bowl with Lemon Parsley Gremolata
4. Old Fashioned Chicken and Rice
5. Marry Me Chicken and Rice (one pan dinner recipe)
6. Chicken and Rice with Sun Dried Tomatoes
7. French Onion Chicken and Rice
8. Greek Chicken and Rice

You can find the recipes for these dishes by following the links provided in the search results.


In [15]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='905b39fb-5176-4a6d-ad78-d7fa3788b531'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ns8hy1nee', 'function': {'arguments': '{"query":"chicken and rice recipes"}', 'name': 'web_search'}, 'type': 'function'}, {'id': 't3me1sgk0', 'function': {'arguments': '{"query":"ideas for leftover chicken"}', 'name': 'web_search'}, 'type': 'function'}, {'id': 'pt3f32hv6', 'function': {'arguments': '{"query":"chicken and rice dish ideas"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 282, 'total_tokens': 337, 'completion_time': 0.063841647, 'completion_tokens_details': None, 'prompt_time': 0.030539618, 'prompt_tokens_details': None, 'queue_time': 0.282609871, 'total_time': 0.094381265}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e